In [97]:
%pip install catboost

In [98]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error, mean_absolute_percentage_error, r2_score
from catboost import CatBoostRegressor, Pool

In [99]:
default_params = {
    'loss_function': 'MAE',
    'eval_metric': 'MAE',
    'bootstrap_type': 'Bernoulli',
    'random_seed': 42,
    'max_ctr_complexity': 1,
    'verbose': False
}

In [100]:
dem_params = {
    **default_params,
    'iterations': 754,
    'subsample': 0.8,
    'rsm': 0.6,
    'random_strength': 4,
    'depth': 4,
    'learning_rate': 0.06
}

market_params = {
    **default_params,
    'iterations': 1299,
    'subsample': 0.4,
    'rsm': 0.5,
    'random_strength': 7,
    'depth': 4,
    'learning_rate': 0.03
}

market_zip_params = {
    **default_params,
    'iterations': 350,
    'subsample': 0.8,
    'rsm': 0.65,
    'random_strength': 15,
    'depth': 3,
    'learning_rate': 0.08,
    'cat_features': ['REGION']
    }

market_dem_params = {
    **default_params,
    'iterations': 1422,
    'subsample': 0.6,
    'rsm': 0.6,
    'random_strength': 7,
    'depth': 4,
    'learning_rate': 0.03
}

market_zip_dem_params = {
    **default_params,
    'iterations': 1090,
    'subsample': 0.9,
    'rsm': 0.65,
    'random_strength': 5,
    'depth': 4,
    'learning_rate': 0.03,
    'cat_features': ['REGION']
    }

In [101]:
dem_model = CatBoostRegressor(**dem_params)
market_model = CatBoostRegressor(**market_params)
market_zip_model = CatBoostRegressor(**market_zip_params)
market_dem_model = CatBoostRegressor(**market_dem_params)
market_zip_dem_model = CatBoostRegressor(**market_zip_dem_params)

In [102]:
# Function to split data into train and test
# Returns X_train, X_test, y_train, y_test
def split_data(df):
    test_set = ['2025-09-01', '2025-10-01', '2025-11-01', '2025-12-01', '2026-01-01', '2026-02-01', '2026-03-01']

    train = df[~df['PERIOD_BEGIN'].isin(test_set)].copy()
    test = df[df['PERIOD_BEGIN'].isin(test_set)].copy()

    y_train = train.pop('AVG_SALE_TO_LIST')
    X_train = train.drop(columns=['PERIOD_BEGIN', 'PERIOD_END'])
    y_test = test.pop('AVG_SALE_TO_LIST')
    X_test = test.drop(columns=['PERIOD_BEGIN', 'PERIOD_END'])

    return X_train, X_test, y_train, y_test

In [103]:
market_df = pd.read_csv('market.csv')
market_zip_df = pd.read_csv('market_zip.csv')
market_dem_df = pd.read_csv('market_dem.csv')
market_zip_dem_df = pd.read_csv('market_zip_dem.csv')
dem_df = market_dem_df.drop(columns=['MEDIAN_LIST_PRICE', 'MEDIAN_LIST_PPSF', 'HOMES_SOLD', 'PENDING_SALES', 'NEW_LISTINGS', 'INVENTORY', 'MEDIAN_DOM', 'OFF_MARKET_IN_TWO_WEEKS'])

In [104]:
def train_and_evaluate(model, df):
    X_train, X_test, y_train, y_test = split_data(df)
    model.fit(X_train, y_train)

    y_preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_preds)
    mape = mean_absolute_percentage_error(y_test, y_preds)
    mse = mean_squared_error(y_test, y_preds)
    rmse = root_mean_squared_error(y_test, y_preds)
    me = np.mean(y_test - y_preds)
    r2 = r2_score(y_test, y_preds)

    fi = model.get_feature_importance()
    features = model.feature_names_
    cat_cols = set(model.get_cat_feature_indices())

    test_pool = Pool(data=X_test, label=y_test, cat_features=list(cat_cols))

    shap_vals = model.get_feature_importance(test_pool, type='ShapValues')
    shap_vals = shap_vals[:, :-1]

    shap_df = pd.DataFrame(shap_vals, columns=features, index=X_test.index)

    # Mean_Abs_SHAP, Max_Upside, Max_Downside, and Direction_Corr
    rows = []
    for idx, col in enumerate(features):
        s = shap_df[col]

        if idx in cat_cols or not pd.api.types.is_numeric_dtype(X_test[col]):
            corr = np.nan
        else:
            corr = X_test[col].corr(s)

        rows.append({
            'Feature': col,
            'Importance_pct': fi[idx],
            'Mean_Abs_SHAP': s.abs().mean(),
            'Max_Upside': s.max(),
            'Max_Downside': s.min(),
            'Direction_Corr': corr
        })

    summary = pd.DataFrame(rows).sort_values(by='Importance_pct', ascending=False)

    print(f'MAE: {mae}')
    print(f'MSE: {mse}')
    print(f'RMSE: {rmse}')
    print(f'ME: {me}')
    print(f'R2: {r2}')
    print(summary)
    return summary

In [105]:
market_summary = train_and_evaluate(market_model, market_df)

MAE: 0.011435033778088907
MSE: 0.0002494785749436839
RMSE: 0.015794890786063824
ME: -0.0016435570195830506
R2: 0.5131931869631503
                   Feature  Importance_pct  Mean_Abs_SHAP  Max_Upside  \
6               MEDIAN_DOM       36.158404       0.009593    0.016643   
7  OFF_MARKET_IN_TWO_WEEKS       27.648558       0.006436    0.014755   
1         MEDIAN_LIST_PPSF       12.643844       0.004473    0.013981   
0        MEDIAN_LIST_PRICE        6.811961       0.000568    0.004029   
2               HOMES_SOLD        5.969446       0.002366    0.011674   
5                INVENTORY        4.864261       0.002733    0.007452   
3            PENDING_SALES        3.615645       0.001075    0.003593   
4             NEW_LISTINGS        2.287880       0.000894    0.002065   

   Max_Downside  Direction_Corr  
6     -0.039216       -0.844294  
7     -0.027320        0.983357  
1     -0.014949       -0.813859  
0     -0.023750        0.221966  
2     -0.012045        0.935253  
5     -0

In [106]:
market_dem_summary = train_and_evaluate(market_dem_model, market_dem_df)

MAE: 0.010756016763616007
MSE: 0.00022560663643206189
RMSE: 0.015020207602828328
ME: -0.0030925413531585288
R2: 0.5597744307051336
                               Feature  Importance_pct  Mean_Abs_SHAP  \
6                           MEDIAN_DOM       33.308554       0.009490   
7              OFF_MARKET_IN_TWO_WEEKS       19.974218       0.005072   
1                     MEDIAN_LIST_PPSF        6.856692       0.002849   
12                          median_age        6.312959       0.001845   
0                    MEDIAN_LIST_PRICE        3.709947       0.000892   
8                     total_population        3.598113       0.000986   
20              pct_hispanic_or_latino        2.895810       0.001187   
11             pct_bachelors_or_higher        2.094637       0.000499   
14              pct_black_non_hispanic        2.028695       0.000615   
2                           HOMES_SOLD        2.010278       0.001310   
13              pct_white_non_hispanic        1.946413       0.000

In [107]:
market_zip_summary = train_and_evaluate(market_zip_model, market_zip_df)

MAE: 0.011228154235810243
MSE: 0.0002385310474935383
RMSE: 0.015444450378486711
ME: -0.003958557869295023
R2: 0.5345550652320231
                   Feature  Importance_pct  Mean_Abs_SHAP  Max_Upside  \
7               MEDIAN_DOM       38.097619       0.008978    0.016849   
0                   REGION       28.628785       0.006229    0.011960   
8  OFF_MARKET_IN_TWO_WEEKS       17.632790       0.004768    0.010896   
2         MEDIAN_LIST_PPSF        6.054535       0.002544    0.008956   
1        MEDIAN_LIST_PRICE        2.634328       0.000493    0.001987   
3               HOMES_SOLD        2.425563       0.000936    0.006761   
5             NEW_LISTINGS        2.012645       0.001145    0.003827   
4            PENDING_SALES        1.361298       0.000553    0.001418   
6                INVENTORY        1.152439       0.001022    0.002654   

   Max_Downside  Direction_Corr  
7     -0.033206       -0.844813  
0     -0.022827             NaN  
8     -0.014923        0.973386  
2   

In [108]:
market_zip_dem_summary = train_and_evaluate(market_zip_dem_model, market_zip_dem_df)

MAE: 0.010850640636752803
MSE: 0.00022759736618394196
RMSE: 0.015086330441294926
ME: -0.003992890472076199
R2: 0.5558899255673712
                               Feature  Importance_pct  Mean_Abs_SHAP  \
7                           MEDIAN_DOM       31.396831       0.009514   
0                               REGION       19.032797       0.005154   
8              OFF_MARKET_IN_TWO_WEEKS       17.075712       0.004572   
2                     MEDIAN_LIST_PPSF        5.015183       0.001853   
13                          median_age        3.590709       0.000852   
9                     total_population        3.009128       0.001027   
1                    MEDIAN_LIST_PRICE        2.499885       0.000560   
3                           HOMES_SOLD        2.000281       0.001240   
12             pct_bachelors_or_higher        1.745836       0.000512   
14              pct_white_non_hispanic        1.699763       0.000517   
15              pct_black_non_hispanic        1.539652       0.0003

In [109]:
dem_summary = train_and_evaluate(dem_model, dem_df)

MAE: 0.01801644470748098
MSE: 0.0005001287932038031
RMSE: 0.02236355949315321
ME: -0.014681283322552545
R2: 0.024100149752467437
                               Feature  Importance_pct  Mean_Abs_SHAP  \
4                           median_age       20.580024       0.004673   
0                     total_population        9.255254       0.003020   
12              pct_hispanic_or_latino        8.563369       0.002091   
2                         poverty_rate        7.142656       0.002566   
3              pct_bachelors_or_higher        7.119450       0.001851   
8               pct_asian_non_hispanic        6.975406       0.001305   
13                           sex_ratio        6.549302       0.001641   
11  pct_two_or_more_races_non_hispanic        6.454202       0.001174   
5               pct_white_non_hispanic        5.955695       0.001968   
6               pct_black_non_hispanic        5.239192       0.001037   
1              median_household_income        4.955395       0.00110